# Fixing the Spectral Index

In this tutorial, we will show how to initialise an analysis instance with a fixed spectral index. 

In [ ]:
import numpy as np

To start with, the prcedure to begin the analysis is to create a ``Config`` instance, collecting the public dataset collections and to define the source position to carry out the point source analysis. This serves as the backbone for carrying out analysis via SkyLLH. 

In [2]:
from skyllh.core.config import Config

cfg = Config()

In [ ]:
from skyllh.datasets.i3.PublicData_14y_ps import create_dataset_collection

dsc = create_dataset_collection(cfg=cfg)
print(dsc.dataset_names)
datasets = dsc['IC40', 'IC59', 'IC79', 'IC86_I-XI']

The source chosen for this example is NGC 1068. 

In [4]:
from skyllh.analyses.i3.publicdata_ps.time_integrated_ps import create_analysis
from skyllh.core.source_model import PointLikeSource

src_ra = 40.67  # degrees
src_dec = -0.01  # degrees

source = PointLikeSource(ra=np.radians(src_ra), dec=np.radians(src_dec))

In [7]:
ana = create_analysis(cfg=cfg, datasets=datasets, source=source, refplflux_gamma=2.0)

100%|██████████| 136/136 [00:00<00:00, 2999.89it/s]


This now initialises the analysis instance with a fiexed power law spectral index $\gamma = 2.0$. 

Consequently, the ``calculate_fluxmodel_scaling_factor()`` method of the analysis now returns an dimentionless scaling factor. It does not take the model parameters as input anymore because it is bound to the flux model and energy range that are defined for the analysis.
But if the ``energy_range`` property is updated, the flux normalisation will also be updated correctly. You just can't change the flux parameters (gamma in this case) anymore.

In [8]:
ana.energy_range = (1e2, 1e9)
print(ana.calculate_fluxmodel_scaling_factor())
ana.energy_range = (1e2, 1e3)
print(ana.calculate_fluxmodel_scaling_factor())
ana.energy_range = (1e5, 1e6)
print(ana.calculate_fluxmodel_scaling_factor())

3.0837040854223075e-17
2.004341561536863e-15
1.041381138284864e-16


Further utility functions are available, 
1. ``ana.mu2flux(mu)`` returns a flux normalization at the chosen spectral index, chosen during the initialisation of the analysis instance. 
2. ``ana.flux2mu(flux_norm)`` returns the mean number of signal events for the given flux normalization value at the chosen spectral index.

In [ ]:
from skyllh.core.random import RandomStateService

ana.energy_range = None

rss = RandomStateService(seed=1)
(ts, x, status) = ana.unblind(rss)
print(f'TS = {ts:.3f}')
print(f'ns = {x["ns"]:.2f}')
print('(gamma is fixed at 2.0 — it does not appear in the fit result dict)')

With a fixed spectral index, only ``ns`` is a free parameter.
The flux normalization is therefore **uniquely determined** from ``ns`` without any ambiguity — there is no gamma to scan over.
``ana.mu2flux(ns)`` returns the flux at the reference energy (1 TeV by default), and ``ana.flux2mu(flux)`` inverts that mapping.

In [ ]:
ns_bf = x['ns']

flux_norm = ana.mu2flux(ns_bf)
print(f'Flux normalization at E_0=1 TeV: {flux_norm:.3e} GeV^-1 cm^-2 s^-1')

ns_roundtrip = ana.flux2mu(flux_norm)
print(f'Roundtrip ns (flux2mu ∘ mu2flux): {ns_roundtrip:.4f}  (should equal {ns_bf:.4f})')

Compare with the free-gamma result from the ``fitting_a_source`` tutorial (gamma ≈ 3.21, ns ≈ 80, flux scaling factor ≈ 2.87e-14):
fixing gamma = 2.0 forces a harder spectrum assumption, which changes the fitted ns and the corresponding flux normalization substantially.
For a source like NGC 1068 whose true spectrum is soft, a fixed hard index can lead to a poor fit — the free-gamma analysis is generally preferable unless there is strong prior knowledge of the spectral shape.